# 第 7 章 · QoS：通信质量设置

本章回答：同一个话题，相机图像要"最新的、丢几帧无所谓"，控制指令要"一条都不能丢"，地图要"后来的人也能拿到"，这些怎么设？两边设置不一样会怎样？为什么有时节点明明在、话题名也对，却收不到消息？

配套讲义：仓库里 `ros2/notes/lessons/07-qos.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 ROS 2 和 DDS。兼容规则就是两张表（讲义第 2 步），这里把表写成"给每个选项排个高低"的函数，再把讲义里两张表逐行跑一遍。截止时间 / 租期的比较规则讲义里有，这里不演示（练习 7-3 自己判断）。

In [ ]:
from collections import deque
from dataclasses import dataclass

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 Keep last | 深度 5 只留最近 5 条 | `keep_last([...8 条], 5)` | scan4…scan8 |
| 第 2 步 可靠性表 | 要求不高于提供 | `compatible(QoS(...), QoS(...))` | 只有 BE 发 + REL 收 ❌ |
| 第 2 步 持久性表 | 同上 | 同上 | 只有 Volatile 发 + Transient local 收 ❌ |
| 第 2 步 地图的坑 | 晚来的 Volatile 收不到旧消息 | `map_pub.late_join(...)` | Volatile：[]；Transient local：['map v1'] |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**常量 `DURABILITY`**

In [ ]:
DURABILITY = {"VOLATILE": 0, "TRANSIENT_LOCAL": 1}

**类 `QoS`**：下面要用到的小工具。

In [ ]:
@dataclass
class QoS:
    reliability: str = "RELIABLE"  # 默认配置：Keep last 10、Reliable、Volatile
    durability: str = "VOLATILE"
    depth: int = 10

**常量 `RELIABILITY`**

In [ ]:
RELIABILITY = {"BEST_EFFORT": 0, "RELIABLE": 1}  # 数字越大 = 质量越高

**函数 `compatible`**：订阅者"要求"的每一项都不高于发布者"提供"的，才能连上。

In [ ]:
def compatible(pub: QoS, sub: QoS) -> bool:
    """订阅者"要求"的每一项都不高于发布者"提供"的，才能连上。"""
    return (RELIABILITY[sub.reliability] <= RELIABILITY[pub.reliability]
            and DURABILITY[sub.durability] <= DURABILITY[pub.durability])

**类 `Publisher`**：Transient local 的发布者会留着最近 depth 条，后来加入的 Transient local 订阅者也能拿到。

In [ ]:
class Publisher:
    """Transient local 的发布者会留着最近 depth 条，后来加入的 Transient local 订阅者也能拿到。"""

    def __init__(self, qos: QoS):
        self.qos = qos
        self.kept = deque(maxlen=qos.depth)

    def publish(self, msg):
        if self.qos.durability == "TRANSIENT_LOCAL":
            self.kept.append(msg)

    def late_join(self, sub: QoS) -> list:
        """一个订阅者在消息发完以后才加入，它一开始能收到哪些旧消息。"""
        if not compatible(self.qos, sub):
            return []
        return list(self.kept) if sub.durability == "TRANSIENT_LOCAL" else []

**常量 `SENSOR_DATA`**

In [ ]:
SENSOR_DATA = QoS(reliability="BEST_EFFORT", depth=5)  # Sensor data 预设：Best effort、较小队列

**函数 `keep_last`**：History = Keep last、Depth = N：没处理完的只留最近 N 条。

In [ ]:
def keep_last(messages: list, depth: int) -> list:
    """History = Keep last、Depth = N：没处理完的只留最近 N 条。"""
    return list(deque(messages, maxlen=depth))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · Keep last + Depth：攒了 8 条没处理，深度 5 只留最近 5 条")
print("  ", keep_last([f"scan{i}" for i in range(1, 9)], 5))

In [ ]:
print("\n第二步 · 可靠性兼容表（讲义第 2 步）")
for p in RELIABILITY:
    for s in RELIABILITY:
        ok = compatible(QoS(reliability=p), QoS(reliability=s))
        print(f"  发布者 {p:<11} 订阅者 {s:<11} → {'✅' if ok else '❌'}")

In [ ]:
print("\n第二步（续）· 持久性兼容表")
for p in DURABILITY:
    for s in DURABILITY:
        ok = compatible(QoS(durability=p), QoS(durability=s))
        print(f"  发布者 {p:<15} 订阅者 {s:<15} → {'✅' if ok else '❌'}")

In [ ]:
print("\n第二步（再续）· 地图的坑：地图只发一次，晚来的订阅者能不能拿到")
map_pub = Publisher(QoS(durability="TRANSIENT_LOCAL", depth=1))
map_pub.publish("map v1")
print("  晚来的 Volatile 订阅者拿到:", map_pub.late_join(QoS(durability="VOLATILE")))
print("  晚来的 Transient local 订阅者拿到:", map_pub.late_join(QoS(durability="TRANSIENT_LOCAL", depth=1)))

In [ ]:
print("\n第三步 · 预设配置长什么样")
print("  默认:", QoS())
print("  Sensor data:", SENSOR_DATA)

## 3. 代码里才看得到的

1. **兼容规则就是"给选项排高低再比大小"**：把 Best effort 记 0、Reliable 记 1，订阅者的数不大于发布者的就能连。
2. **截止时间、租期这里没写**：规则是"订阅者要求的时长不能比提供的短"，留给练习 7-3 自己判断。
3. **连不上时真 ROS 2 不报错退出**，只发一个 incompatible QoS 事件、打一条警告；模拟里 `compatible` 返回 False 就等于收不到。

## 4. 自己改一改

- 把 `map_pub` 的 `depth` 改成 3，连发三版地图，看晚来的 Transient local 订阅者拿到几版；
- 给 `QoS` 加一个 `history` 字段（Keep last / Keep all），想一想 Keep all 时 `keep_last` 该怎么改。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **七策略**：历史、深度、可靠、持久，截止、寿命、活性。
- **匹配**：订阅者要求，发布者提供；要求不高于提供才成交，不成交不报错、只是收不到。
- **常用搭配**：传感器尽力而为（Best effort）；指令可靠（Reliable）；地图、静态信息两边都 Transient local。
- **排查**：收不到消息，先 `topic info -v` 看 QoS。